In [1]:
%load_ext autoreload

In [2]:
from jsonargparse import CLI, set_parsing_settings

import os
import sys
import time
import warnings
from pathlib import Path
from pprint import pprint
from typing import Any, Dict, Iterator, List, Literal, Optional, Tuple, Union

import torch
from transformers import AutoTokenizer, TextStreamer

/capstor/scratch/cscs/jkirchen/daint_210_129_nightly_singleshot/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# NOTE: hf version requires a conversion to exist at the target location
MASK_ID = None
MIN_MASK_ID = None
MAX_MASK_ID = None

HF_MODEL_GPU = 1
LITGPT_MODEL_GPU = 2

# when run from Daint
# MODEL_PATH = "/capstor/store/cscs/userlab/lp98/jkirchen/singleshot-root/singleshot/checkpoints/meta-llama/Meta-Llama-3.1-8B"
# MODEL_PATH = "/capstor/store/cscs/userlab/lp98/jkirchen/singleshot-root/singleshot/outputs/daint_prod_q4/daint_prod_q4_32N128n_12cacdc9/latest"
# when run from Tuo
# MODEL_PATH = "/p/vast1/kirchenb/singleshot-root/singleshot/checkpoints/meta-llama/Meta-Llama-3.1-8B"
# MODEL_PATH = "/p/vast1/kirchenb/singleshot-root/singleshot/outputs/debug_metamath_full_rand_k2-8_ex_valk/debug_metamath_full_rand_k2-8_ex_valk_ss_L3-8B_pqds-cfgfc55c3da_toks11.6B_ss-cfg2059c710_mb32-wb128-sl160_lr-2000steps-const1e-05tonull_1N4n/latest"
# MODEL_PATH = "/p/vast1/kirchenb/singleshot-root/singleshot/outputs/debug_metamath_full_rand_k2-8_ex_valk/debug_metamath_full_rand_k2-8_ex_valk_baseline/latest" # tuo fixed k baseline
# MODEL_PATH = "/p/vast1/kirchenb/daint/daint_prod_q4/daint_prod_q4_32N128n_e3c83f91/latest" # daint fixed k
# MODEL_PATH = "/p/vast1/kirchenb/daint/daint_prod_q4/daint_prod_q4_32N128n_12cacdc9/latest" # daint rand k
# MODEL_PATH = "/p/vast1/kirchenb/daint/daint_prod_q4/daint_prod_q4_128N512n_fd7261ea/latest" # daint fixed k, big data
# k2-16 ablation
# MODEL_PATH = "/capstor/store/cscs/userlab/lp98/jkirchen/singleshot-root/singleshot/outputs/daint_prod_q1_k2-16_abl/daint_prod_q1_k2-16_abl_1N4n_81607a4b/step-00230000" # fixed k
# MODEL_PATH = "/capstor/store/cscs/userlab/lp98/jkirchen/singleshot-root/singleshot/outputs/daint_prod_q1_k2-16_abl/daint_prod_q1_k2-16_abl_1N4n_fa344ca4/step-00230000" # rand k
# MASK_ID = 128002
# EOS_ID = 128001

# IFT ablation metamath

# New L3 Magpie trained on chat fmt metamath
# MODEL_PATH = "/capstor/scratch/cscs/jkirchen/singleshot-root/singleshot/outputs/daint_prod_ift_mask_fix/daint_prod_ift_mask_fix_1N4n_9d30cad5/step-00000000"
MODEL_PATH = "/capstor/scratch/cscs/jkirchen/singleshot-root/singleshot/outputs/daint_prod_ift_mask_fix/daint_prod_ift_mask_fix_1N4n_9d30cad5/step-00100160"
MODEL_CLASS = "llama"
MASK_ID = 128259
# EOS_ID = 128001
# EOS_ID = [128009,128001]
EOS_ID = [128009,128001, 48]
# MIN_MASK_ID = 128259
# MAX_MASK_ID = 128259+15

# New Qwen3-8B-Base trained on bos only metamath
# # MODEL_PATH = "/capstor/scratch/cscs/jkirchen/singleshot-root/singleshot/outputs/daint_prod_ift_mask_fix/daint_prod_ift_mask_fix_1N4n_3c08f241/step-00000000" 
# MODEL_PATH = "/capstor/scratch/cscs/jkirchen/singleshot-root/singleshot/outputs/daint_prod_ift_mask_fix/daint_prod_ift_mask_fix_1N4n_3c08f241/step-00100160"
# MODEL_CLASS = "qwen3"
# MASK_ID = 151669
# EOS_ID = 151643

# # Qwen3-4B-Instruct-2507 trained on bos only metamath, max k 16
# # MODEL_PATH = "/capstor/scratch/cscs/jkirchen/singleshot-root/singleshot/outputs/daint_prod_ift_q3-4b/daint_prod_ift_q3-4b_1N4n_16cdce0f/step-00000000" 
# # MODEL_PATH = "/capstor/scratch/cscs/jkirchen/singleshot-root/singleshot/outputs/daint_prod_ift_q3-4b/daint_prod_ift_q3-4b_1N4n_16cdce0f/step-00030000"
# MODEL_PATH = "/capstor/scratch/cscs/jkirchen/singleshot-root/singleshot/outputs/daint_prod_ift_q3-4b/daint_prod_ift_q3-4b_1N4n_16cdce0f/step-00100160"
# MODEL_CLASS = "qwen3"
# MASK_ID = 151669
# # EOS_ID = [151645,151643]
# EOS_ID = [151645,151643,48]

# # Qwen3-4B-Instruct-2507 trained on chat fmt metamath, max k 16
# # MODEL_PATH = "/capstor/scratch/cscs/jkirchen/singleshot-root/singleshot/outputs/daint_prod_ift_q3-4b/daint_prod_ift_q3-4b_1N4n_d2aa1bac/step-00000000" 
# # MODEL_PATH = "/capstor/scratch/cscs/jkirchen/singleshot-root/singleshot/outputs/daint_prod_ift_q3-4b/daint_prod_ift_q3-4b_1N4n_d2aa1bac/step-00030000"
# MODEL_PATH = "/capstor/scratch/cscs/jkirchen/singleshot-root/singleshot/outputs/daint_prod_ift_q3-4b/daint_prod_ift_q3-4b_1N4n_d2aa1bac/step-00100160"
# MODEL_CLASS = "qwen3"
# MASK_ID = 151669
# EOS_ID = 151645
# # EOS_ID = 151643
# # EOS_ID = 48

# # New Qwen3-4B-Instruct-2507 trained on bos only metamath, max k 8
# # MODEL_PATH = "/capstor/scratch/cscs/jkirchen/singleshot-root/singleshot/outputs/daint_prod_ift_q3-4b/daint_prod_ift_q3-4b_1N4n_ccfee559/step-00000000" 
# # MODEL_PATH = "/capstor/scratch/cscs/jkirchen/singleshot-root/singleshot/outputs/daint_prod_ift_q3-4b/daint_prod_ift_q3-4b_1N4n_ccfee559/step-00030000" 
# MODEL_PATH = "/capstor/scratch/cscs/jkirchen/singleshot-root/singleshot/outputs/daint_prod_ift_q3-4b/daint_prod_ift_q3-4b_1N4n_ccfee559/step-00103476" 
# MODEL_CLASS = "qwen3"
# MASK_ID = 151669
# # EOS_ID = 151645
# EOS_ID = 151643
# # EOS_ID = 48

# # New Qwen3-4B-Instruct-2507 trained on chat fmt metamath, max k 8
# # MODEL_PATH = "/capstor/scratch/cscs/jkirchen/singleshot-root/singleshot/outputs/daint_prod_ift_q3-4b/daint_prod_ift_q3-4b_1N4n_426eff81/step-00000000" 
# # MODEL_PATH = "/capstor/scratch/cscs/jkirchen/singleshot-root/singleshot/outputs/daint_prod_ift_q3-4b/daint_prod_ift_q3-4b_1N4n_426eff81/step-00030000" 
# MODEL_PATH = "/capstor/scratch/cscs/jkirchen/singleshot-root/singleshot/outputs/daint_prod_ift_q3-4b/daint_prod_ift_q3-4b_1N4n_426eff81/step-00103476" 
# MODEL_CLASS = "qwen3"
# MASK_ID = 151669
# # EOS_ID = 151645
# # EOS_ID = 151643
# EOS_ID = [151645,151643]
# # EOS_ID = 48

# IFT ablation magpie

# New L3 Magpie trained on chat fmt magpie
# # MODEL_PATH = "/capstor/scratch/cscs/jkirchen/singleshot-root/singleshot/outputs/daint_prod_ift_magpie/daint_prod_ift_magpie_1N4n_44004b35/step-00000000" 
# # MODEL_PATH = "/capstor/scratch/cscs/jkirchen/singleshot-root/singleshot/outputs/daint_prod_ift_magpie/daint_prod_ift_magpie_1N4n_44004b35/step-00125881"
# New L3 Magpie trained on bos only magpie
# # MODEL_PATH = "/capstor/scratch/cscs/jkirchen/singleshot-root/singleshot/outputs/daint_prod_ift_magpie/daint_prod_ift_magpie_1N4n_e379bd46/step-00000000" 
# MODEL_PATH = "/capstor/scratch/cscs/jkirchen/singleshot-root/singleshot/outputs/daint_prod_ift_magpie/daint_prod_ift_magpie_1N4n_e379bd46/step-00125881"
# MODEL_CLASS = "llama"
# MASK_ID = 128259
# # EOS_ID = 128001
# EOS_ID = 48 # hardcoding "Q:" to get an accurate measurement


# Ent crusher tests
# MODEL_PATH = "/capstor/scratch/cscs/jkirchen/singleshot-root/singleshot/outputs/daint_prod_ent_crusher/daint_prod_ent_crusher_1N4n_0eb278f3/latest"
# MODEL_CLASS = "llama"
# MASK_ID = 128259
# EOS_ID = [128009,128001, 48]

# MODEL_PATH = "/capstor/scratch/cscs/jkirchen/singleshot-root/singleshot/outputs/daint_prod_ent_crusher/daint_prod_ent_crusher_1N4n_44ec004b/latest"
# MODEL_CLASS = "llama"
# MASK_ID = 128259
# EOS_ID = [128009,128001, 48]

# # Unique mask token test
# MODEL_PATH = "/capstor/scratch/cscs/jkirchen/singleshot-root/singleshot/outputs/daint_prod_unique_masks/daint_prod_unique_masks_1N4n_25dbd712/step-00050000"
# MODEL_CLASS = "llama"
# MASK_ID = 128259
# # EOS_ID = 128001
# # EOS_ID = [128009,128001]
# EOS_ID = [128009,128001, 48]
# MIN_MASK_ID = 128259
# MAX_MASK_ID = 128259+15



# Whether to use the base AutoModel classed included in transformers
# which also allows for the hub based remote code version too.
# USE_BASE_AUTOMODEL = True
USE_BASE_AUTOMODEL = False


# quicker debug time
# MODEL_PATH = "/p/vast1/kirchenb/singleshot-root/singleshot/checkpoints/meta-llama/Llama-3.2-1B"

HF_MODEL_PATH = MODEL_PATH

# # hub based version, make sure to have pushed an updated set of model files!
# USE_BASE_AUTOMODEL = True
# HF_MODEL_PATH = "tomg-group-umd/debug_metamath_full_rand_k2-8_ex_valk_baseline_latest"
# HF_MODEL_PATH = "tomg-group-umd/daint_prod_q4_32N128n_e3c83f91_latest"
# HF_MODEL_PATH = "tomg-group-umd/daint_prod_q4_32N128n_12cacdc9_latest"
# HF_MODEL_PATH = "tomg-group-umd/daint_prod_q4_128N512n_fd7261ea_latest"
# k2-16 ablation
# HF_MODEL_PATH = "jwkirchenbauer/daint_prod_q1_k2-16_abl_1N4n_fa344ca4_step-00230000"

CHECKPOINT_DIR = Path(MODEL_PATH)
HF_CHECKPOINT_DIR = Path(HF_MODEL_PATH)

# PRECISION = "bf16-true"
PRECISION = "32-true"
# COMPILE = True
COMPILE = False

# from metamath based validation data/wandb runs
# PROMPT = f"""\
# Given $g(x) = x^2$ and $f(x) = 2x - 1$, what is the value of $f(g(2))$?

# First, we need to find $g(2)$.
# Since $g(x) = x^2$, we have $g(2) = \
# """
# PROMPT = f"""\
# Sandy plans to paint one wall in her bedroom. The wall is 9 feet high and 12 feet long. There is a 2-foot by 4-foot area on that wall that she will not have to paint due to the window. How many square feet will she need to paint?

# The total area of the wall is\
# """
# PROMPT = f"""\
# Scott, Mary, and Ken have set a goal of raising $4,000 for their local children's cancer hospital. Currently, Mary's collection is five times the amount Ken has, and three times the amount Scott has. If Ken currently has $600, how much have the three exceeded their goal by?

# If Ken currently has $600, then Mary's collection is 5 * $600 = $3000.
# And Scott's collection is\
# """
# PROMPT = f"""\
# Given $g(x) = x^2$ and $f(x) = 3x - 1$, what is the value of $f(g(2))$?

# First, we need to find $g(2)$.
# Since $g(x) = x^2$, we have $g(2) = \
# """
# PROMPT = f"""\
# Sandy plans to paint one wall in her bedroom. The wall is 9 feet high and 12 feet long. There is a 2-foot by 4-foot area on that wall that she will not have to paint due to the window. How many square feet will she need to paint?\
# """
# PROMPT = f"""\
# Given $g(x) = x^2$ and $f(x) = 2x - 1$, what is the value of $f(g(2))$?\
# """
# PROMPT = f"""\
# Q: Claire makes a 3 egg omelet every morning for breakfast.  How many dozens of eggs will she eat in 4 weeks?\nA: Let's think step by step.\
# """

# from gsm8k cot test itself
# PROMPT = "" # no fewshot
PROMPT = f"""\
Q: There are 15 trees in the grove. Grove workers will plant trees in the grove today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?\nA: Let's think step by step. There are 15 trees originally. Then there were 21 trees after some more were planted. So there must have been 21 - 15 = 6. The answer is 6.\n\nQ: If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?\nA: Let's think step by step. There are originally 3 cars. 2 more cars arrive. 3 + 2 = 5. The answer is 5.\n\nQ: Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?\nA: Let's think step by step. Originally, Leah had 32 chocolates. Her sister had 42. So in total they had 32 + 42 = 74. After eating 35, they had 74 - 35 = 39. The answer is 39.\n\nQ: Jason had 20 lollipops. He gave Denny some lollipops. Now Jason has 12 lollipops. How many lollipops did Jason give to Denny?\nA: Let's think step by step. Jason started with 20 lollipops. Then he had 12 after giving some to Denny. So he gave Denny 20 - 12 = 8. The answer is 8.\n\nQ: Shawn has five toys. For Christmas, he got two toys each from his mom and dad. How many toys does he have now?\nA: Let's think step by step. Shawn started with 5 toys. If he got 2 toys each from his mom and dad, then that is 4 more toys. 5 + 4 = 9. The answer is 9.\n\nQ: There were nine computers in the server room. Five more computers were installed each day, from monday to thursday. How many computers are now in the server room?\nA: Let's think step by step. There were originally 9 computers. For each of 4 days, 5 more computers were added. So 5 * 4 = 20 computers were added. 9 + 20 is 29. The answer is 29.\n\nQ: Michael had 58 golf balls. On tuesday, he lost 23 golf balls. On wednesday, he lost 2 more. How many golf balls did he have at the end of wednesday?\nA: Let's think step by step. Michael started with 58 golf balls. After losing 23 on tuesday, he had 58 - 23 = 35. After losing 2 more, he had 35 - 2 = 33 golf balls. The answer is 33.\n\nQ: Olivia has $23. She bought five bagels for $3 each. How much money does she have left?\nA: Let's think step by step. Olivia had 23 dollars. 5 bagels for 3 dollars each will be 5 x 3 = 15 dollars. So she has 23 - 15 dollars left. 23 - 15 is 8. The answer is 8.\n\n\
""" # with fewshot
# PROMPT += f"""\
# Q: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?\nA:\
# """
# PROMPT += f"""\
# Q: A robe takes 2 bolts of blue fiber and half that much white fiber.  How many bolts in total does it take?\nA:\
# """
# PROMPT += f"""\
# Q: Josh decides to try flipping a house.  He buys a house for $80,000 and then puts in $50,000 in repairs.  This increased the value of the house by 150%.  How much profit did he make?\nA:\
# """
PROMPT += f"""\
Q: Claire makes a 3 egg omelet every morning for breakfast.  How many dozens of eggs will she eat in 4 weeks?\nA: Let's think step by step.\
"""
# PROMPT = f"""\
# Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?\n\nSince Janet's ducks lay 16 eggs per day and she eats three for breakfast and\
# """
# PROMPT = f"""\
# A robe takes 2 bolts of blue fiber and half that much white fiber.  How many bolts in total does it take?\n\n Half of two bolts\
# """
# PROMPT = f"""\
# Josh decides to try flipping a house.  He buys a house for $80,000 and then puts in $50,000 in repairs.  This increased the value of the house by 150%.  How much profit did he make?\n\nA 150% increase in value is\
# """
# PROMPT = f"""\
# Here's a pattern, just repeat after me: I am a great language model. I am a great language model. I am\
# """
# PROMPT = f"""\
# Here's a pattern, just repeat after me: I am a great language model. I am a great language model. I am\
# """
# PROMPT = f"""\
# Four score and seven years ago \
# """
# PROMPT = f"""\
# What happened in Tiananmen Square? \
# """
# PROMPT = f"""\
# Here's what Tom Goldstein did last week, he first  \
# """
# PROMPT = f"""\
# I am planning a trip to Japan, and I would like thee to write an itinerary for my journey in a Shakespearean style. You are not allowed to use any commas in your response.\n\nFirst ye will \
# """
# PROMPT = f"""\
# <|begin_of_text|>Write a poem about how I am missing my classes. The poem must have 4 sections marked with SECTION X. Finish the poem with this exact phrase: "Can I get my money back for the classes I missed?"\n\nSECTION 1\n \
# """
# PROMPT = f"""\
# <|begin_of_text|>Write a story of exactly 2 paragraphs about a man who wakes up one day and realizes that he's inside a video game. Separate the paragraphs with the markdown divider: ***\n\n \
# """
# PROMPT = f"""\
# <|begin_of_text|>I'm going on a camping trip and I want to make sure I have the best equipment possible. What are some must-haves for a beginner?\n\n \
# """
# PROMPT = f"""\
# <|begin_of_text|>Come up with a special pattern of numbers, and then start repeating it.\n\n \
# """


DO_CHAT_TEMPLATE = False
# DO_CHAT_TEMPLATE = True


# MAX_NEW_TOKENS = 32
# MAX_NEW_TOKENS = 64
# MAX_NEW_TOKENS = 96
MAX_NEW_TOKENS = 128
# MAX_NEW_TOKENS = 256
# MAX_NEW_TOKENS = 1024

# hf only control
DO_MTP = True
# DO_MTP = False

# controls both
# K_TOKS=1
# K_TOKS=2
# K_TOKS=3
# K_TOKS=6
# K_TOKS=8
# K_TOKS=10
# K_TOKS=12
K_TOKS=16
# K_TOKS=32
# K_TOKS=64


STRATEGY = None
# STRATEGY = ("conf_adapt", 0.99)
# STRATEGY = ("conf_adapt", 0.95)
STRATEGY = ("conf_adapt", 0.9)
# STRATEGY = ("conf_adapt", 0.8)
# STRATEGY = ("conf_adapt", 0.7)
# STRATEGY = ("conf_adapt", 0.6)
# STRATEGY = ("conf_adapt", 0.3)
# STRATEGY = ("conf_adapt", 0.1)
# STRATEGY = ("random", [1/K_TOKS for _ in range(K_TOKS)]) # uniform
# STRATEGY = ("random", [0.5]+[0 for _ in range(K_TOKS-2)]+[0.5]) # 1 and K_TOKS bimodal
# STRATEGY = ("random", [1/K_TOKS]+[0 for _ in range(K_TOKS-2)]+[(K_TOKS-1)/K_TOKS]) # 1 and K_TOKS bimodal
# STRATEGY = ("random", [0.05]+[0 for _ in range(K_TOKS-2)]+[0.95]) # 1 and K_TOKS bimodal
# STRATEGY = ("random", [0.02]+[0 for _ in range(K_TOKS-2)]+[0.98]) # 1 and K_TOKS bimodal
# STRATEGY = ("conf_adapt_sample@1", 0.9, 0.1) # we include a temperature here
# STRATEGY = ("conf_adapt_sample@1", 0.9, 0.2)
# STRATEGY = ("conf_adapt_sample@1", 0.9, 0.9)
# STRATEGY = ("conf_adapt_sample@1", 0.9, 1.0)
# STRATEGY = ("conf_adapt_sample@1", 0.9, 0.0)

SEED = 1234
# SEED = 4321
# SEED = 1823
# SEED = 9669

## HF based generation setup

In [4]:
%autoreload 2

import random
import numpy as np
import torch

from transformers import AutoModelForCausalLM
from transformers import GenerationConfig

from litgpt.transformers_local.llama.modeling_llama import LlamaForCausalLM
from litgpt.transformers_local.qwen3.modeling_qwen3 import Qwen3ForCausalLM

def seed_everything_hf(seed):
    # python RNG
    random.seed(seed)
    # pytorch RNGs
    torch.manual_seed(seed)
    # torch.backends.cudnn.deterministic = True
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # numpy RNG
    np.random.seed(seed)

@torch.inference_mode()
def load_hf_model(
    checkpoint_dir: Path,
    precision: Optional[str] = None,
    compile: bool = False,
):
    if USE_BASE_AUTOMODEL:
        # hub and standard
        
        model = AutoModelForCausalLM.from_pretrained(
            checkpoint_dir,
            trust_remote_code=True,
            token=os.environ.get("HF_HUB_TOKEN", None),
            dtype={
                "16-true": torch.float16,
                "bf16-true": torch.bfloat16,
                "32-true": torch.float32,
            }[precision],
            attn_implementation="sdpa",
        )
    else:
        # local debugging implementation    
        if MODEL_CLASS == "llama":
            class_ = LlamaForCausalLM
        elif MODEL_CLASS == "qwen3":
            class_ = Qwen3ForCausalLM
        else:
            raise ValueError(f"Unknown MODEL_CLASS+{MODEL_CLASS}")
        model = class_.from_pretrained(
            checkpoint_dir,
            dtype={
                "16-true": torch.float16,
                "bf16-true": torch.bfloat16,
                "32-true": torch.float32,
            }[precision],
            attn_implementation="sdpa",
        )

    model.to(torch.device(f"cuda:{HF_MODEL_GPU}" if torch.cuda.is_available() else "cpu"))
    print(f"Model loaded to use attn_implementation: {model.config._attn_implementation} on device {model.device} in dtype {model.dtype}")
    if compile:
        model = torch.compile(model)
    model.eval()
    return model

@torch.inference_mode()
def generate_response_hf(
    prompt: str = "What food do llamas eat?",
    max_new_tokens: int = None,
    model = None, 
    tokenizer = None,
    streamer = None,
) -> None:
    
    if model.device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(model.device)
    
    encoded = tokenizer.encode(prompt, return_tensors="pt").to(model.device)
    prompt_length = encoded.size(1)
    
    max_returned_tokens = prompt_length + max_new_tokens    

    seed_everything_hf(SEED)
    
    t0 = time.perf_counter()
    output_ids = model.generate(
        input_ids=encoded,
        max_new_tokens=max_new_tokens,
        generation_config=GenerationConfig(
            do_sample=False,
            # temperature=0.0,
            # top_p=1.0,
        ),
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
        streamer=streamer,
        # mtp codepath additional args
        do_mtp=DO_MTP,
        max_returned_tokens=max_returned_tokens,
        k_toks=K_TOKS,
        mask_id=MASK_ID,
        min_mask_id=MIN_MASK_ID,
        max_mask_id=MAX_MASK_ID,
        eos_id=EOS_ID,
        include_prompt=True,
        strategy=STRATEGY,
        return_mtp_result_dict=True,
    )
    t = time.perf_counter() - t0
    
    # y = output_ids[0, prompt_length:]
    y = output_ids["token_ids"][0, prompt_length:]
    
    # print(f"#### BEGIN Prompt ####")
    # print(f"{prompt}")
    # print(f"########")
    # print(f"tokenized as:\n{encoded.squeeze(0).tolist()}")
    # print(f"#### END Prompt ####")
    # print(f"#### BEGIN Generation ####")
    # print(f"{tokenizer.decode(y, skip_special_tokens=False)}")
    # print(f"########")
    # print(f"tokenized as:\n{y.tolist()}")
    # print(f"#### END Generation ####")
    # tokens_generated = y.size(0)
    # print(
    #     f"Time for inference: {t:.02f} sec total, {tokens_generated / t:.02f} tokens/sec over {tokens_generated} tokens."
    # )
    if model.device.type == "cuda":
        max_memory_allocated_per_gpu = torch.cuda.max_memory_allocated(model.device) / 1024**3
        max_memory_reserved_per_gpu = torch.cuda.max_memory_reserved(model.device) / 1024**3
        torch.cuda.reset_peak_memory_stats(model.device)
        print(
            f"Memory alloc/reserved: {max_memory_allocated_per_gpu:.02f} GB / {max_memory_reserved_per_gpu:.02f} GB",
            flush=True
        )
    return y, encoded, output_ids

In [5]:
hf_tokenizer = AutoTokenizer.from_pretrained(HF_CHECKPOINT_DIR)
streamer = TextStreamer(hf_tokenizer)

# try:
#     del model
# except:
#     pass
# try:
#     del hf_model
# except:
#     pass
hf_model = load_hf_model(
    checkpoint_dir=HF_CHECKPOINT_DIR,
    precision=PRECISION,
    compile=COMPILE,
)

The tokenizer you are loading from '/capstor/scratch/cscs/jkirchen/singleshot-root/singleshot/outputs/daint_prod_ift_mask_fix/daint_prod_ift_mask_fix_1N4n_9d30cad5/step-00100160' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Loading local LlamaForCausalLM!
Model loaded to use attn_implementation: sdpa on device cuda:1 in dtype torch.float32


## HF based generation execute

In [7]:
FMTD_PROMPT = PROMPT

def format_query_for_chat_generation(query, do_tokenize=False):
        messages = [
            {"role": "user", "content": query},
        ]
        return hf_tokenizer.apply_chat_template(messages, tokenize=do_tokenize, add_generation_prompt=True)

if DO_CHAT_TEMPLATE:
    FMTD_PROMPT = format_query_for_chat_generation(PROMPT, do_tokenize=False)

hf_generation, hf_prompt_encoding, mtp_dict = generate_response_hf(
    prompt=FMTD_PROMPT, 
    max_new_tokens=MAX_NEW_TOKENS,
    tokenizer=hf_tokenizer,
    model=hf_model,
    streamer=streamer,
)
print(f"hf_prompt_encoding: {hf_prompt_encoding.squeeze(0).cpu().tolist()}", flush=True)
try:
    print(f"hf prompt tokens == litgpt prompt tokens: {torch.equal(hf_prompt_encoding.squeeze(0).cpu(), litgpt_prompt_encoding.squeeze(0).cpu())}", flush=True)
    print(f"hf generation tokens == litgpt generation tokens: {torch.equal(hf_generation.cpu(), litgpt_generation.cpu())}", flush=True)
except:
    pass

Executing custom MTP generation codepath!

<BEGIN Streaming Prompt>
Q: There are 15 trees in the grove. Grove workers will plant trees in the grove today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?
A: Let's think step by step. There are 15 trees originally. Then there were 21 trees after some more were planted. So there must have been 21 - 15 = 6. The answer is 6.

Q: If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?
A: Let's think step by step. There are originally 3 cars. 2 more cars arrive. 3 + 2 = 5. The answer is 5.

Q: Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?
A: Let's think step by step. Originally, Leah had 32 chocolates. Her sister had 42. So in total they had 32 + 42 = 74. After eating 35, they had 74 - 35 = 39. The answer is 39.

Q: Jason had 20 lollipops. He gave Denny some lollipops. Now Jason has 12 lollipop

step. Claire eats 3 eggs a day. In 4 weeks, she will eat 3 x 4 x 7 = 84 eggs. 84 eggs is 84 / 12 = 7 dozens. The answer is 7.

Q:

<END Streaming Generation>
Using a total of 1+1+21 forward evals, time for prefill plus first/compilation step (1+1) was 0.33 sec, generation time was 0.75 sec @ 65.06 tokens/sec steady state over 49 tokens.
Strategy used: ('conf_adapt', 0.9)
Average effective k_toks over generation: 2.22, full array of effective k_toks: [1, 1, 3, 1, 2, 1, 1, 7, 3, 1, 1, 4, 4, 2, 1, 2, 2, 4, 1, 1, 5, 1, 2]
Memory alloc/reserved: 30.53 GB / 30.76 GB
hf_prompt_encoding: [48, 25, 2684, 527, 220, 868, 12690, 304, 279, 10707, 588, 13, 41234, 7487, 690, 6136, 12690, 304, 279, 10707, 588, 3432, 13, 4740, 814, 527, 2884, 11, 1070, 690, 387, 220, 1691, 12690, 13, 2650, 1690, 12690, 1550, 279, 10707, 588, 7487, 6136, 3432, 5380, 32, 25, 6914, 596, 1781, 3094, 555, 3094, 13, 2684, 527, 220, 868, 12690, 13517, 13, 5112, 1070, 1051, 220, 1691, 12690, 1306, 1063, 810, 1051, 39441, 13, 21

In [ ]:
# print(mtp_dict)
raw_token_ids = mtp_dict['token_ids'][0][hf_prompt_encoding.shape[1]:].tolist()
eff_k_arr = mtp_dict['effective_k_values']

def print_chunked_output(token_ids, eff_k_values, tokenizer=hf_tokenizer):
    """eff_k_values is a list/array whose sum is the same length as token_ids, indicating the effective k , or chunk size used at each generation step.
    THis prints out the token ids chunked according to the effective k values.
    """
    print(f"Chunked output decoded:")
    idx = 0
    all_chunks = []
    for step, k in enumerate(eff_k_values):
        chunk = token_ids[idx:idx+k]
        decoded_chunk = tokenizer.decode(chunk, skip_special_tokens=False)
        # print(f"Step {step}: effective k={k}, decoded chunk: '{decoded_chunk}'")
        all_chunks.append(f"step {step} @ k={k} | {decoded_chunk}")
        idx += k
    print("\n".join(all_chunks))
    
print_chunked_output(raw_token_ids, eff_k_arr)

## LitGPT based generation setup

In [ ]:
import lightning as L
from lightning.fabric.plugins import BitsandbytesPrecision

from litgpt.config import Config
from litgpt.model import GPT, Block
from litgpt.prompts import PromptStyle, has_prompt_style, load_prompt_style
from litgpt.tokenizer import Tokenizer
from litgpt.utils import (
    _BITANDBYTES_AVAILABLE_NOT_EQUAL_0_42_0,
    check_file_size_on_cpu_and_warn,
    check_valid_checkpoint_dir,
    extend_checkpoint_dir,
    get_default_supported_precision,
    load_checkpoint,
    check_nvlink_connectivity,
    _TORCH_EQUAL_2_7,
    _TORCH_EQUAL_2_8,
)

In [ ]:
%autoreload 2

from litgpt.generate.base_mtp import (
    generate_mtp,
    reset_model_kv_cache,
)


In [ ]:
@torch.inference_mode()
def load_model(
    checkpoint_dir: Path = None,
    *,
    quantize: Optional[
        Literal["bnb.nf4", "bnb.nf4-dq", "bnb.fp4", "bnb.fp4-dq", "bnb.int8"]
    ] = None,
    precision: Optional[str] = None,
    compile: bool = False,
) -> Tuple[L.Fabric, GPT, Tokenizer, PromptStyle]:

    torch.set_float32_matmul_precision("high")

    checkpoint_dir = extend_checkpoint_dir(checkpoint_dir)
    pprint(locals())

    precision = precision or get_default_supported_precision(training=False)

    plugins = None
    if quantize is not None and quantize.startswith("bnb."):
        if "mixed" in precision:
            raise ValueError("Quantization and mixed precision is not supported.")
        if _BITANDBYTES_AVAILABLE_NOT_EQUAL_0_42_0:
            warnings.warn(
                "LitGPT only supports bitsandbytes v0.42.0. This may result in errors when using quantization."
            )
        dtype = {
            "16-true": torch.float16,
            "bf16-true": torch.bfloat16,
            "32-true": torch.float32,
        }[precision]
        plugins = BitsandbytesPrecision(quantize[4:], dtype)
        precision = None

    fabric = L.Fabric(devices=[LITGPT_MODEL_GPU], precision=precision, plugins=plugins)

    check_valid_checkpoint_dir(checkpoint_dir)
    config = Config.from_file(checkpoint_dir / "model_config.yaml")

    checkpoint_path = checkpoint_dir / "lit_model.pth"
    check_file_size_on_cpu_and_warn(checkpoint_path, fabric.device)

    tokenizer = Tokenizer(checkpoint_dir)
    prompt_style = (
        load_prompt_style(checkpoint_dir)
        if has_prompt_style(checkpoint_dir)
        else PromptStyle.from_config(config)
    )

    print(
        f"Loading model {str(checkpoint_path)!r} with {config.__dict__}",
        file=sys.stderr,
    )
    t0 = time.perf_counter()
    with fabric.init_module(empty_init=True):
        # model = GPT(config)
        model = GPT(config, hparams=None, use_block_mask=False)
    print(
        f"Time to instantiate model: {time.perf_counter() - t0:.02f} seconds.",
        file=sys.stderr,
    )
    # with fabric.init_tensor():
    #     # set the max_seq_length to limit the memory usage to what we need
    #     model.max_seq_length = max_returned_tokens
    #     # enable the kv cache
    #     model.set_kv_cache(batch_size=1)
    
    model.eval()

    if compile:
        # torch._dynamo.config.automatic_dynamic_shapes = True
        # torch._inductor.config.triton.unique_kernel_names = True
        # torch._inductor.config.coordinate_descent_tuning = True
        # global next_token
        # next_token = torch.compile(next_token, mode="reduce-overhead")
        torch._dynamo.config.cache_size_limit = 256
        model = torch.compile(model)

    model = fabric.setup_module(model)

    t0 = time.perf_counter()
    load_checkpoint(fabric, model, checkpoint_path)

    # work around PyTorch issue https://github.com/pytorch/pytorch/issues/152162
    # which does not like the lazy initialization to be called in dynamo.
    # TODO: Happens with PyTorch 2.7+
    if (
        (_TORCH_EQUAL_2_7 or _TORCH_EQUAL_2_8)
        and (model._forward_module.__class__.__name__ == "OptimizedModule")
        and (
            model._forward_module._orig_mod.__class__.__name__
            == "FullyShardedDataParallel"
        )
    ):
        from torch.distributed.fsdp._runtime_utils import _root_pre_forward

        _root_pre_forward(
            model._forward_module._orig_mod, model._forward_module._orig_mod, [], {}
        )
        _root_pre_forward(
            model_teacher._forward_module._orig_mod,
            model_teacher._forward_module._orig_mod,
            [],
            {},
        )

    fabric.print(
        f"Time to load the model weights: {time.perf_counter() - t0:.02f} seconds.",
        file=sys.stderr,
    )

    fabric.print(f"Model on device: {model.device}.")
    return (fabric, model, tokenizer, prompt_style)

@torch.inference_mode()
def generate_response(
    prompt: str = "What food do llamas eat?",
    include_prompt = True,
    # add_explicit_bos: Optional[str] = None,
    add_explicit_bos: Optional[str] = "<|begin_of_text|>",
    sys_prompt: Optional[str] = None,
    num_samples: int = 1,
    max_new_tokens: int = None,
    k_toks: int = 1,
    mask_id: Optional[int] = None,
    eos_id: Optional[int] = None,
    fabric = None, 
    model = None, 
    tokenizer = None, 
    streamer = None,
    prompt_style = None,
    strategy: Optional[tuple] = None,
) -> None:
    
    if model.device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(model.device)

    if add_explicit_bos is not None:
        prompt = f"{add_explicit_bos}{prompt}"
    prompt = prompt_style.apply(prompt, sys_prompt=sys_prompt)
    encoded = tokenizer.encode(prompt, device=fabric.device)
    prompt_length = encoded.size(0)
    
    max_returned_tokens = prompt_length + max_new_tokens
    reset_model_kv_cache(model=model, batch_size=1, max_returned_tokens=max_returned_tokens, device=fabric.device, fabric=fabric)
    
    L.seed_everything(SEED)
    t0 = time.perf_counter()
    y = generate_mtp(
        model=model,
        prompt=encoded,
        max_returned_tokens=max_returned_tokens,
        k_toks=k_toks,
        eos_id=eos_id,
        include_prompt=include_prompt,
        tokenizer=tokenizer,
        streamer=streamer,
        strategy=strategy,
    )
    t = time.perf_counter() - t0

    if include_prompt:
        y = y[prompt_length:]
    
    tokens_generated = y.size(0)
    
    print(f"\n\n#### Complete Output ####")
    print(f"\n\n#### BEGIN Prompt ####")
    print(f"{prompt}")
    print(f"########")
    print(f"tokenized as:\n{encoded.tolist()}")
    print(f"#### END Prompt ####")
    print(f"#### BEGIN Generation ####")
    print(f"{tokenizer.decode(y, skip_special_tokens=False)}")
    print(f"########")
    print(f"tokenized as:\n{y.tolist()}")
    print(f"#### END Generation ####")
    print(
        f"Time for inference: {t:.02f} sec total, {tokens_generated / t:.02f} tokens/sec over {tokens_generated} tokens."
    )
    if model.device.type == "cuda":
        max_memory_allocated_per_gpu = torch.cuda.max_memory_allocated(model.device) / 1024**3
        max_memory_reserved_per_gpu = torch.cuda.max_memory_reserved(model.device) / 1024**3
        torch.cuda.reset_peak_memory_stats(model.device)
        print(
            f"Memory alloc/reserved: {max_memory_allocated_per_gpu:.02f} GB / {max_memory_reserved_per_gpu:.02f} GB",
            flush=True
        )
    
    return y, encoded


In [ ]:
hf_tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
streamer = TextStreamer(hf_tokenizer)

# try:
#     del model
# except:
#     pass
# try:
#     del hf_model
# except:
#     pass

fabric, model, tokenizer, prompt_style = load_model(
    checkpoint_dir=CHECKPOINT_DIR,
    precision=PRECISION,
    compile=COMPILE,
)

## LitGPT based generation execute

In [ ]:
litgpt_generation, litgpt_prompt_encoding = generate_response(
    prompt=PROMPT, 
    k_toks=K_TOKS,
    mask_id=MASK_ID,
    eos_id=EOS_ID,
    max_new_tokens=MAX_NEW_TOKENS,
    fabric=fabric, 
    model=model, 
    tokenizer=tokenizer, 
    prompt_style=prompt_style,
    streamer=streamer,
    strategy=STRATEGY,
)
try:
    print(f"hf prompt tokens == litgpt prompt tokens: {torch.equal(hf_prompt_encoding.squeeze(0).cpu(), litgpt_prompt_encoding.squeeze(0).cpu())}", flush=True)
    print(f"hf generation tokens == litgpt generation tokens: {torch.equal(hf_generation.cpu(), litgpt_generation.cpu())}", flush=True)
except:
    pass

In [ ]:
# arr = [2, 2, 1, 4, 4, 1, 1, 1, 3, 5, 2, 1, 4, 1, 1, 2, 1, 7, 1, 1, 1, 1, 5, 1]
# avg_k = sum(arr)/len(arr)
# print(f"avg_k: {avg_k}")

# no seed ->       bakes muffins for her friends every day with four, she sells 16 - 3 - 4 = 9 eggs at the farmers' market. Since she sells each egg for $2, she makes 9 x 2 = 18 dollars every day at the farmers' market
# w/ sample 0.2 -> bakes muffins for her friends every day with four, she sells 16 - 3 - 4 = 9 eggs at the farmers' market. Since she sells each egg for $2, she makes 9 x 2 = 18 dollars every day at the farmers' market
# w/ sample 1.0 -> bakes muffins for her friends every day with four, she sells 16 - 3 - 4 = 9 eggs at the farmers' market. Since she sells each egg for $2, she makes 9 \times 2 = 18 dollars every day at the farmers' market
# new seeed
# w/ sample 1.0 -> bakes muffins for her friends every day with four, she has 16 - 3 - 4 = 9 eggs left to sell at the farmers' market. Since she sells the remainder for $2 per fresh duck egg, she makes 9 x 2 = 18 dollars

#### analyze output discrepancy if relevant

In [ ]:

# analyze the divergence point between hf_generation and litgpt_generation
if not torch.equal(hf_generation.cpu(), litgpt_generation.cpu()):
    #Find divergence point
    min_len = min(hf_generation.size(0), litgpt_generation.size(0))
    divergence = 0
    for i in range(min_len):
        if hf_generation[i] != litgpt_generation[i]:
            print(f"Divergence at position {i}: hf={hf_generation[i]}, litgpt={litgpt_generation[i]}", flush=True)
            divergence = i
            break
    # add special token at position and then print rest of sequence
    print(f"hf generation tokens: {hf_generation[max(divergence-5, 0):divergence+5].cpu().tolist()}", flush=True)
    print(f"litgpt generation tokens: {litgpt_generation[max(divergence-5, 0):divergence+5].cpu().tolist()}", flush=True)

    print(hf_generation)
    print(litgpt_generation)
    
    print("#### HF Generation in text: ####")
    print(hf_tokenizer.decode(hf_generation, skip_special_tokens=False))
    print("#### LitGPT Generation in text: ####") 
    print(tokenizer.decode(litgpt_generation, skip_special_tokens=False))

## Special token debugging

In [ ]:
from transformers import AutoTokenizer
from pathlib import Path



# TOK_PATH = ""
# TOK_PATH = "checkpoints/extended/meta-llama/Meta-Llama-3.1-8B-Instruct-MTPV128384"
# TOK_PATH = "checkpoints/extended/allenai/Llama-3.1-Tulu-3-8B-SFT-MTPV128384"
# TOK_PATH = "checkpoints/extended/Magpie-Align/Llama-3.1-8B-Magpie-Align-SFT-v0.1-MTPV128384"
# TOK_PATH = "checkpoints/extended/nvidia/OpenMath2-Llama3.1-8B-MTPV128384"
# TOK_PATH = "checkpoints/extended/Qwen/Qwen3-8B-Base-MTP"
# TOK_PATH = "checkpoints/extended/Qwen/Qwen3-8B-MTP"
TOK_PATH = "checkpoints/extended/Qwen/Qwen3-4B-Instruct-2507-MTP"

tokenizer = AutoTokenizer.from_pretrained(TOK_PATH)


In [ ]:

def format_chat(example, do_tokenize=False):
        messages = [
            {"role": "user", "content": example["query"]},
            {"role": "assistant", "content": example["response"]},
        ]
        return {
            "text": tokenizer.apply_chat_template(
                messages, tokenize=do_tokenize
            )
        }

example = {
    "query": "Hello world!",
    "response": "Hi there! How can I assist you today?"
}
formatted = format_chat(example, do_tokenize=False)
print(formatted["text"])

In [ ]:
# TGT_STR = ""
# TGT_STR = "<|eot_id|><|start_header_id|>user<|end_header_id|>"
# TGT_STR = "Q: There are 15 trees in the grove."
# TGT_STR = "A: There are 15 trees in the grove."
TGT_STR = "Q:"
# TGT_STR = "<|endoftext|>"

"-".join([str(i) for i  in tokenizer.encode(TGT_STR, add_special_tokens=False)])